# NYC Yellow Taxi, March 2024: Load, Clean and Explore in PostgreSQL

## 0. Setup

In [3]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pgserver') is None:
  subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pgserver', 'sqlalchemy>=2.0,<2.1', 'psycopg2-binary', 'pandas', 'pyarrow'],
    check=True,
  )

In [5]:
import os
import urllib.request
import zipfile

import pandas as pd
from pgserver.postgres_server import get_server
from sqlalchemy import create_engine, text

pg = get_server('/tmp/pg_data')
engine = create_engine(pg.get_uri(), isolation_level='AUTOCOMMIT', connect_args={'options': '-c timezone=UTC'})
conn = engine.connect()

conn.exec_driver_sql('SELECT version()').scalar()

'PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit'

In [6]:
def run_sql(sql: str) -> None:
  conn.execute(text(sql))


def query(sql: str) -> pd.DataFrame:
  result = conn.execute(text(sql))

  return pd.DataFrame(result.fetchall(), columns=list(result.keys()))


def copy_csv(table: str, path: str, options: str = 'FORMAT csv, HEADER true') -> int:
  raw_conn = engine.raw_connection()
  cursor = raw_conn.cursor()
  with open(path, encoding='utf-8') as csv_file:
    cursor.copy_expert(f'COPY {table} FROM STDIN WITH ({options})', csv_file)
  loaded_rows = cursor.rowcount
  raw_conn.commit()
  raw_conn.close()

  return loaded_rows

## 1. Data source

In [26]:
DATA_DIR = 'data'
PARQUET_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet'
ZONES_URL = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
PARQUET_PATH = os.path.join(DATA_DIR, 'yellow_tripdata_2024-03.parquet')
CSV_PATH = os.path.join(DATA_DIR, 'hw3_taxi_sample.csv')
ZONES_PATH = os.path.join(DATA_DIR, 'taxi_zone_lookup.csv')
SAMPLE_SIZE = 100_000
COLUMNS = [
  'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance',
  'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount',
  'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
  'congestion_surcharge', 'Airport_fee',
]

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ZONES_PATH):
  urllib.request.urlretrieve(ZONES_URL, ZONES_PATH)
if not os.path.exists(CSV_PATH):
  if not os.path.exists(PARQUET_PATH):
    urllib.request.urlretrieve(PARQUET_URL, PARQUET_PATH)
  month = pd.read_parquet(PARQUET_PATH, columns=COLUMNS)
  month.sample(n=SAMPLE_SIZE, random_state=42).sort_values('tpep_pickup_datetime').to_csv(CSV_PATH, index=False)

pd.read_csv(CSV_PATH, nrows=3)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-03-01 00:00:04,2024-03-01 00:29:13,1.0,5.77,1.0,N,186,181,1,31.0,1.0,0.5,9.00,0.0,1.0,45.00,2.5,0.0
1,2,2024-03-01 00:00:18,2024-03-01 00:16:56,2.0,7.56,1.0,N,230,244,1,31.7,1.0,0.5,7.34,0.0,1.0,44.04,2.5,0.0
2,2,2024-03-01 00:00:27,2024-03-01 00:09:21,1.0,2.06,1.0,N,48,239,1,12.1,1.0,0.5,2.00,0.0,1.0,19.10,2.5,0.0


## 2. Staging and COPY

In [28]:
run_sql('''
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;
DROP TABLE IF EXISTS hw3_taxi_staging CASCADE;
DROP TABLE IF EXISTS hw3_taxi_zones CASCADE;
DROP TABLE IF EXISTS hw3_taxi_zones_staging CASCADE;

CREATE TABLE hw3_taxi_staging (
  vendor_id_raw TEXT,
  pickup_raw TEXT,
  dropoff_raw TEXT,
  passenger_count_raw TEXT,
  trip_distance_raw TEXT,
  rate_code_raw TEXT,
  store_and_fwd_raw TEXT,
  pu_location_id_raw TEXT,
  do_location_id_raw TEXT,
  payment_type_raw TEXT,
  fare_amount_raw TEXT,
  extra_raw TEXT,
  mta_tax_raw TEXT,
  tip_amount_raw TEXT,
  tolls_amount_raw TEXT,
  improvement_surcharge_raw TEXT,
  total_amount_raw TEXT,
  congestion_surcharge_raw TEXT,
  airport_fee_raw TEXT
);

CREATE TABLE hw3_taxi_zones_staging (
  location_id_raw TEXT,
  borough_raw TEXT,
  zone_raw TEXT,
  service_zone_raw TEXT
);
''')

copy_options = "FORMAT csv, HEADER true, DELIMITER ',', NULL '\\N'"
{
  'trips': copy_csv('hw3_taxi_staging', CSV_PATH, copy_options),
  'zones': copy_csv('hw3_taxi_zones_staging', ZONES_PATH, copy_options),
}

{'trips': 100000, 'zones': 265}

In [30]:
query('''
SELECT COUNT(*) AS loaded_rows
FROM hw3_taxi_staging
''')

,loaded_rows
0,100000


## 3. Typed table

| Column | Meaning | Source | Type | Cleaning |
|---|---|---|---|---|
| `trip_id` | surrogate trip key | none | `BIGINT IDENTITY` | generated, because the file has no natural id |
| `vendor_id` | TPEP data provider (1 Creative Mobile, 2 Curb, 6 Myle) | integer | `SMALLINT` | `NULLIF(TRIM(...), '')`; `CHECK IN (1, 2, 6, 7)` |
| `pickup_at`, `dropoff_at` | pickup and drop-off moments | New York local time without zone, `2024-03-01 00:18:51` | `TIMESTAMPTZ` | `::TIMESTAMP AT TIME ZONE 'America/New_York'` to keep the absolute moment; `NOT NULL` |
| `pickup_date` | calendar date of the pickup in New York | derived | `DATE` | date taken from the local time before localization, for per-day grouping |
| `passenger_count` | passenger count entered by the driver | float `1.0` or empty | `SMALLINT` | `::NUMERIC::SMALLINT`; empty → `NULL`; `CHECK 0..9` |
| `trip_distance` | taximeter distance, miles | float | `NUMERIC(10, 2)` | `CHECK >= 0`; hundredths of a mile are enough |
| `rate_code_id` | rate code (1 standard, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 negotiated, 6 group) | float or empty | `SMALLINT` | marker `99` ("Null/unknown" in the TLC dictionary) → `NULL`; `CHECK 1..6` |
| `store_and_fwd` | whether the record was stored in the vehicle without a connection | `Y` / `N` / empty | `BOOLEAN` | `Y` → `TRUE`, `N` → `FALSE`, empty → `NULL` |
| `pu_location_id`, `do_location_id` | TLC pickup and drop-off zones | integer | `INTEGER` | `NOT NULL`, FK to `hw3_taxi_zones` |
| `payment_type` | payment type (0 flex fare, 1 card, 2 cash, 3 no charge, 4 dispute, 5 unknown, 6 voided) | integer | `SMALLINT` | `NOT NULL`, `CHECK 0..6` |
| `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `congestion_surcharge`, `airport_fee`, `total_amount` | fare components and total in dollars | float | `NUMERIC(10, 2)` | money to the cent; negative values are kept because they are adjustments and refunds, flagged separately |
| `is_negative_amount` | adjustment-receipt flag | derived | `BOOLEAN` | `TRUE` if `fare_amount` or `total_amount` < 0 |

The zone lookup `hw3_taxi_zones` (`location_id INTEGER PK`, `borough`, `zone`, `service_zone` as `TEXT`) is loaded from the official `taxi_zone_lookup.csv`. The markers `Unknown` and `N/A` in it are replaced with `NULL`.

In [59]:
run_sql('''
CREATE TABLE hw3_taxi_zones (
  location_id INTEGER PRIMARY KEY,
  borough TEXT,
  zone TEXT,
  service_zone TEXT
);

INSERT INTO hw3_taxi_zones (location_id, borough, zone, service_zone)
SELECT
  location_id_raw::INTEGER,
  NULLIF(NULLIF(TRIM(borough_raw), 'Unknown'), 'N/A'),
  NULLIF(NULLIF(TRIM(zone_raw), ''), 'N/A'),
  NULLIF(NULLIF(TRIM(service_zone_raw), ''), 'N/A')
FROM hw3_taxi_zones_staging;

CREATE TABLE hw3_taxi_trips (
  trip_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  vendor_id SMALLINT NOT NULL CHECK (vendor_id IN (1, 2, 6, 7)),
  pickup_at TIMESTAMPTZ NOT NULL,
  dropoff_at TIMESTAMPTZ NOT NULL,
  pickup_date DATE NOT NULL,
  passenger_count SMALLINT CHECK (passenger_count BETWEEN 0 AND 9),
  trip_distance NUMERIC(10, 2) NOT NULL CHECK (trip_distance >= 0),
  rate_code_id SMALLINT CHECK (rate_code_id BETWEEN 1 AND 6),
  store_and_fwd BOOLEAN,
  pu_location_id INTEGER NOT NULL REFERENCES hw3_taxi_zones (location_id),
  do_location_id INTEGER NOT NULL REFERENCES hw3_taxi_zones (location_id),
  payment_type SMALLINT NOT NULL CHECK (payment_type BETWEEN 0 AND 6),
  fare_amount NUMERIC(10, 2) NOT NULL,
  extra NUMERIC(10, 2),
  mta_tax NUMERIC(10, 2),
  tip_amount NUMERIC(10, 2),
  tolls_amount NUMERIC(10, 2),
  improvement_surcharge NUMERIC(10, 2),
  congestion_surcharge NUMERIC(10, 2),
  airport_fee NUMERIC(10, 2),
  total_amount NUMERIC(10, 2) NOT NULL,
  is_negative_amount BOOLEAN NOT NULL,
  CHECK (dropoff_at >= pickup_at),
  CHECK (pickup_date = (pickup_at AT TIME ZONE 'America/New_York')::DATE)
);
''')

In [86]:
run_sql('''
WITH parsed AS (
  SELECT
    NULLIF(TRIM(vendor_id_raw), '')::SMALLINT AS vendor_id,
    NULLIF(TRIM(pickup_raw), '')::TIMESTAMP AS pickup_local,
    NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP AS dropoff_local,
    NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT AS passenger_count,
    NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(10, 2) AS trip_distance,
    NULLIF(NULLIF(TRIM(rate_code_raw), '')::NUMERIC::SMALLINT, 99) AS rate_code_id,
    CASE UPPER(TRIM(store_and_fwd_raw)) WHEN 'Y' THEN TRUE WHEN 'N' THEN FALSE END AS store_and_fwd,
    NULLIF(TRIM(pu_location_id_raw), '')::INTEGER AS pu_location_id,
    NULLIF(TRIM(do_location_id_raw), '')::INTEGER AS do_location_id,
    NULLIF(TRIM(payment_type_raw), '')::SMALLINT AS payment_type,
    NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10, 2) AS fare_amount,
    NULLIF(TRIM(extra_raw), '')::NUMERIC(10, 2) AS extra,
    NULLIF(TRIM(mta_tax_raw), '')::NUMERIC(10, 2) AS mta_tax,
    NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10, 2) AS tip_amount,
    NULLIF(TRIM(tolls_amount_raw), '')::NUMERIC(10, 2) AS tolls_amount,
    NULLIF(TRIM(improvement_surcharge_raw), '')::NUMERIC(10, 2) AS improvement_surcharge,
    NULLIF(TRIM(congestion_surcharge_raw), '')::NUMERIC(10, 2) AS congestion_surcharge,
    NULLIF(TRIM(airport_fee_raw), '')::NUMERIC(10, 2) AS airport_fee,
    NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10, 2) AS total_amount
  FROM hw3_taxi_staging
)
INSERT INTO hw3_taxi_trips (
  vendor_id, pickup_at, dropoff_at, pickup_date, passenger_count, trip_distance, rate_code_id,
  store_and_fwd, pu_location_id, do_location_id, payment_type, fare_amount, extra, mta_tax,
  tip_amount, tolls_amount, improvement_surcharge, congestion_surcharge, airport_fee,
  total_amount, is_negative_amount
)
SELECT
  vendor_id,
  pickup_local AT TIME ZONE 'America/New_York',
  dropoff_local AT TIME ZONE 'America/New_York',
  pickup_local::DATE,
  passenger_count,
  trip_distance,
  rate_code_id,
  store_and_fwd,
  pu_location_id,
  do_location_id,
  payment_type,
  fare_amount,
  extra,
  mta_tax,
  tip_amount,
  tolls_amount,
  improvement_surcharge,
  congestion_surcharge,
  airport_fee,
  total_amount,
  fare_amount < 0 OR total_amount < 0
FROM parsed
WHERE pickup_local IS NOT NULL
  AND dropoff_local IS NOT NULL
  AND dropoff_local >= pickup_local
  AND trip_distance >= 0
  AND (passenger_count IS NULL OR passenger_count BETWEEN 0 AND 9);
''')

## 4. Data-quality audit

In [87]:
query('''
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_taxi_staging
UNION ALL
SELECT 'clean', COUNT(*) FROM hw3_taxi_trips
''')

,layer,n_rows
0,staging,100000
1,clean,99994


In [89]:
query('''
SELECT
  COUNT(*) FILTER (WHERE NULLIF(TRIM(pickup_raw), '') IS NULL OR NULLIF(TRIM(dropoff_raw), '') IS NULL) AS missing_timestamps,
  COUNT(*) FILTER (WHERE dropoff_raw::TIMESTAMP < pickup_raw::TIMESTAMP) AS dropoff_before_pickup,
  COUNT(*) FILTER (WHERE trip_distance_raw::NUMERIC < 0) AS negative_distance,
  COUNT(*) FILTER (WHERE tip_amount_raw::NUMERIC < 0) AS negative_tip
FROM hw3_taxi_staging
''')

,missing_timestamps,dropoff_before_pickup,negative_distance,negative_tip
0,0,6,0,5


Staging has 100,000 rows and clean has 99,994. Exactly 6 rows were dropped by the rule `dropoff_local >= pickup_local`: their drop-off is recorded earlier than the pickup, so the trip duration is negative and the row cannot be repaired without the source. There are no missing timestamps and no negative distances. The 5 rows with negative tips are kept because they are part of adjustment receipts, which are marked with the `is_negative_amount` flag.

In [90]:
query('''
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, total_amount)) AS distinct_business_keys,
  COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, total_amount)) AS duplicate_rows,
  COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id)) AS distinct_keys_without_amount,
  COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id)) AS same_trip_rows
FROM hw3_taxi_trips
''')

,total_rows,distinct_business_keys,duplicate_rows,distinct_keys_without_amount,same_trip_rows
0,99994,99994,0,99960,34


In [94]:
query('''
SELECT
  a.trip_id,
  a.pickup_at,
  a.pu_location_id,
  a.do_location_id,
  a.payment_type,
  a.total_amount,
  b.trip_id AS pair_trip_id,
  b.payment_type AS pair_payment_type,
  b.total_amount AS pair_total_amount
FROM hw3_taxi_trips AS a
JOIN hw3_taxi_trips AS b
  ON b.vendor_id = a.vendor_id
  AND b.pickup_at = a.pickup_at
  AND b.dropoff_at = a.dropoff_at
  AND b.pu_location_id = a.pu_location_id
  AND b.trip_id > a.trip_id
ORDER BY a.pickup_at
LIMIT 10
''')

,trip_id,pickup_at,pu_location_id,do_location_id,payment_type,total_amount,pair_trip_id,pair_payment_type,pair_total_amount
0,8717,2024-03-03 22:55:05+00:00,161,233,2,7.00,8718,2,-7.00
1,9270,2024-03-04 03:05:42+00:00,163,261,4,-33.20,9271,4,33.20
2,10297,2024-03-04 17:23:21+00:00,233,97,2,-35.00,10298,2,35.00
3,18963,2024-03-07 05:10:06+00:00,132,132,2,9.75,18964,2,-9.75
4,22173,2024-03-08 04:04:34+00:00,263,145,4,-24.80,22174,4,24.80
5,22440,2024-03-08 07:04:20+00:00,148,231,4,-13.60,22441,4,13.60
6,23048,2024-03-08 15:43:18+00:00,100,100,4,-11.90,23049,4,11.90
7,28358,2024-03-10 00:04:52+00:00,163,100,4,-16.80,28359,4,16.80
8,34989,2024-03-12 01:37:17+00:00,170,68,4,14.30,34990,4,-14.30
9,43725,2024-03-14 21:16:01+00:00,113,113,2,79.00,43726,2,-79.00


The TLC file has no natural key, so the business key is `(vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, total_amount)`: the same vehicle, the same time and route, the same amount. There are 0 exact duplicates by that key. If the amount is removed from the key, 34 rows repeat another trip: these are "receipt + reversal" pairs with identical times and opposite amounts (`7.00` / `-7.00`). So the check does not reveal a duplication bug but the adjustment mechanism. For model training such pairs should be collapsed, otherwise one trip is counted twice.

In [96]:
query('''
SELECT
  COUNT(*) AS total_rows,
  ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_passenger_count,
  ROUND(COUNT(*) FILTER (WHERE rate_code_id IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_rate_code,
  ROUND(COUNT(*) FILTER (WHERE store_and_fwd IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_store_and_fwd,
  ROUND(COUNT(*) FILTER (WHERE congestion_surcharge IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_congestion,
  ROUND(COUNT(*) FILTER (WHERE airport_fee IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_airport_fee,
  ROUND(COUNT(*) FILTER (WHERE fare_amount IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_fare,
  ROUND(COUNT(*) FILTER (WHERE pickup_at IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_pickup
FROM hw3_taxi_trips
''')

,total_rows,pct_null_passenger_count,pct_null_rate_code,pct_null_store_and_fwd,pct_null_congestion,pct_null_airport_fee,pct_null_fare,pct_null_pickup
0,99994,12.01,13.08,12.01,12.01,12.01,0.00,0.00


In [97]:
query('''
SELECT
  COUNT(*) FILTER (WHERE TRIM(passenger_count_raw) = '') AS empty_passenger_count,
  COUNT(*) FILTER (WHERE TRIM(rate_code_raw) = '') AS empty_rate_code,
  COUNT(*) FILTER (WHERE NULLIF(TRIM(rate_code_raw), '')::NUMERIC = 99) AS rate_code_99_marker,
  COUNT(*) FILTER (WHERE TRIM(store_and_fwd_raw) = '') AS empty_store_and_fwd,
  COUNT(*) FILTER (WHERE payment_type_raw = '0') AS payment_type_0,
  COUNT(*) FILTER (WHERE payment_type_raw = '0' AND TRIM(passenger_count_raw) = '') AS payment_0_and_empty_passengers,
  COUNT(*) FILTER (WHERE NULLIF(TRIM(passenger_count_raw), '')::NUMERIC = 0) AS zero_passengers,
  COUNT(*) FILTER (WHERE pu_location_id_raw IN ('264', '265') OR do_location_id_raw IN ('264', '265')) AS unknown_zone_trips
FROM hw3_taxi_staging
''')

,empty_passenger_count,empty_rate_code,rate_code_99_marker,empty_store_and_fwd,payment_type_0,payment_0_and_empty_passengers,zero_passengers,unknown_zone_trips
0,12015,12015,1069,12015,12015,12015,1136,1016


In [99]:
query('''
SELECT location_id_raw, borough_raw, zone_raw, service_zone_raw
FROM hw3_taxi_zones_staging
WHERE borough_raw IN ('Unknown', 'N/A') OR zone_raw IN ('N/A', '') OR service_zone_raw IN ('N/A', '')
''')

,location_id_raw,borough_raw,zone_raw,service_zone_raw
0,264,Unknown,N/A,N/A
1,265,N/A,Outside of NYC,N/A


In clean, `passenger_count`, `store_and_fwd`, `congestion_surcharge` and `airport_fee` are empty in 12.01% of rows, and `rate_code_id` in 13.08%. The reason is visible in staging: all 12,015 empty rows have `payment_type = 0` (flex fare), so these are not random gaps but a separate record type for which the vendor does not send these fields. Another 1,069 rows have `RatecodeID = 99`, the "Null/unknown" marker from the TLC dictionary, so they become `NULL`; that explains the difference 13.08 − 12.01. `passenger_count = 0` occurs 1,136 times. This is not a gap but an implausible value typed by the driver, so it is kept and needs a flag. In the zone lookup, `264 Unknown` and `265 N/A` are also missing-value markers: 1,016 trips reference them, and their borough becomes `NULL`. Time, amount and distance are 100% populated.

## 5. Distributions and anomalies

In [100]:
query('''
SELECT
  COALESCE(rate_code_id::TEXT, 'NULL') AS rate_code_id,
  COUNT(*) AS n_rows,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_taxi_trips
GROUP BY rate_code_id
ORDER BY n_rows DESC, rate_code_id
LIMIT 10
''')

,rate_code_id,n_rows,pct_rows
0,1,82529,82.53
1,NULL,13079,13.08
2,2,3272,3.27
3,5,632,0.63
4,3,265,0.27
5,4,217,0.22


Rate codes are heavily imbalanced: 82.5% of trips use the standard rate and 13.1% have an unknown rate. The rest (JFK, negotiated, Newark, Nassau/Westchester) together make up less than 4.5%, so for a model the rare rate codes should be merged or encoded with frequency in mind. Rate code 6 (group ride) never occurs in the sample.

In [101]:
query('''
SELECT
  'trip_distance' AS metric,
  COUNT(trip_distance) AS n_non_null,
  MIN(trip_distance) AS min_value,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_distance) AS median_value,
  MAX(trip_distance) AS max_value,
  ROUND(AVG(trip_distance), 2) AS avg_value,
  ROUND(STDDEV_SAMP(trip_distance), 2) AS sd_value
FROM hw3_taxi_trips
UNION ALL
SELECT
  'total_amount',
  COUNT(total_amount),
  MIN(total_amount),
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_amount),
  MAX(total_amount),
  ROUND(AVG(total_amount), 2),
  ROUND(STDDEV_SAMP(total_amount), 2)
FROM hw3_taxi_trips
''')

,metric,n_non_null,min_value,median_value,max_value,avg_value,sd_value
0,trip_distance,99994,0.00,1.75,176836.30,6.12,609.39
1,total_amount,99994,-251.00,20.52,1021.99,27.09,22.97


The median distance is 1.75 miles, while the mean is 6.12 with a standard deviation of 609. The mean is pulled up by a few impossible values, up to 176,836 miles, so for distance the median or percentiles should be used. For `total_amount` the median is $20.52, the mean $27.09, and the range runs from −$251 to $1,021.99: negative amounts are refunds, and the right tail is airport and long trips.

In [132]:
query('''
SELECT
  trip_id,
  pickup_at,
  dropoff_at,
  ROUND(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 60, 1) AS duration_min,
  trip_distance,
  pu_location_id,
  do_location_id,
  fare_amount,
  total_amount
FROM hw3_taxi_trips
WHERE trip_distance > 100
   OR total_amount > 500
   OR dropoff_at - pickup_at > INTERVAL '6 hours'
ORDER BY trip_distance DESC
LIMIT 20
''')

,trip_id,pickup_at,dropoff_at,duration_min,trip_distance,pu_location_id,do_location_id,fare_amount,total_amount
0,19232,2024-03-07 12:41:00+00:00,2024-03-07 12:54:00+00:00,13.0,176836.30,263,161,23.28,32.74
1,52120,2024-03-17 01:56:00+00:00,2024-03-17 02:14:00+00:00,18.0,60125.89,249,163,22.12,26.12
2,41993,2024-03-14 11:45:00+00:00,2024-03-14 12:20:00+00:00,35.0,47413.57,42,141,34.54,38.54
3,36827,2024-03-12 19:59:07+00:00,2024-03-13 04:00:00+00:00,480.9,38.05,132,1,167.20,172.45
4,99956,2024-04-01 03:35:06+00:00,2024-04-01 15:45:48+00:00,730.7,26.26,100,226,550.40,555.40
5,92999,2024-03-29 23:06:05+00:00,2024-03-30 21:57:22+00:00,1371.3,22.39,132,14,84.90,90.65
6,43679,2024-03-14 21:06:09+00:00,2024-03-15 20:43:13+00:00,1417.1,18.93,79,132,70.00,115.13
7,76791,2024-03-24 20:42:59+00:00,2024-03-25 04:00:00+00:00,437.0,18.45,132,50,70.00,90.90
8,96214,2024-03-30 23:13:49+00:00,2024-03-31 21:52:52+00:00,1359.1,17.05,132,170,70.00,90.19
9,141,2024-03-01 07:11:03+00:00,2024-03-02 05:40:53+00:00,1349.8,13.19,132,252,52.70,62.65


The domain check looks for trips longer than 100 miles, more expensive than $500, or lasting more than 6 hours. Three distances of 47–177 thousand miles in 13–35 minutes are physically impossible at a normal price of $22–35: this is an odometer fault, so the distance should be replaced with `NULL` by rule and the row excluded from training data for distance or price regression. 54 trips last more than 6 hours but have ordinary distances and prices, and six of them end exactly at midnight New York time. It looks like the meter was left running or the system closed the trip automatically, so the row gets a flag and stays out of duration-model training. Of the two receipts above $500, the 26-mile trip lasting 12 hours ($555.40) looks like a valid hourly booking and I keep it. The $1,021.99 receipt for 3.8 miles and 14 minutes looks like a data-entry error, so it should be sent to the source for verification and excluded from training for now.

## 6. Ten DQL/EDA queries

### Q1

**Business-question:** Which payment types are the most common and what share of trips does each account for?

In [134]:
query('''
SELECT
  payment_type,
  CASE payment_type
    WHEN 0 THEN 'flex fare'
    WHEN 1 THEN 'credit card'
    WHEN 2 THEN 'cash'
    WHEN 3 THEN 'no charge'
    WHEN 4 THEN 'dispute'
    WHEN 5 THEN 'unknown'
    WHEN 6 THEN 'voided'
  END AS payment_label,
  COUNT(*) AS trips,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_trips,
  ROUND(AVG(total_amount), 2) AS avg_total
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY trips DESC
''')

,payment_type,payment_label,trips,pct_trips,avg_total
0,1,credit card,72107,72.11,29.22
1,2,cash,13568,13.57,23.16
2,0,flex fare,12010,12.01,23.30
3,4,dispute,1638,1.64,2.06
4,3,no charge,671,0.67,7.01


**Interpretation:** Card dominates: 72.1% of trips with an average receipt of $29.22. Cash gives 13.6%, and 12.0% falls on flex fare (`payment_type = 0`), the same records that lack passengers and rate code. Disputes and no-charge trips have an average receipt of only $2–7 because they contain reversals with negative amounts.

### Q2

**Business-question:** At which hours of the day (New York time) are there the most trips?

In [136]:
query('''
SELECT
  EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York')::INT AS pickup_hour,
  COUNT(*) AS trips,
  ROUND(AVG(trip_distance), 2) AS avg_distance
FROM hw3_taxi_trips
GROUP BY pickup_hour
ORDER BY trips DESC
LIMIT 5
''')

,pickup_hour,trips,avg_distance
0,18,6948,2.86
1,17,6695,3.03
2,19,6480,3.07
3,20,6086,3.15
4,21,6077,13.15


**Interpretation:** Demand peaks in the evening: 18:00 has 6,948 trips, followed by 17:00 and 19:00. The average distance at 21:00 (13.15 miles) is implausible because the 176,836-mile trip landed there; the median for that hour is only 1.94 miles. This clearly shows why the mean cannot be used as a feature without cleaning outliers.

### Q3

**Business-question:** What share of trips has no passenger count, and does it depend on the vendor?

In [140]:
query('''
SELECT
  vendor_id,
  COUNT(*) AS trips,
  COUNT(*) FILTER (WHERE passenger_count IS NULL) AS missing_passengers,
  ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / COUNT(*), 2) AS pct_missing
FROM hw3_taxi_trips
GROUP BY vendor_id
ORDER BY vendor_id
''')

,vendor_id,trips,missing_passengers,pct_missing
0,1,24184,2946,12.18
1,2,75797,9051,11.94
2,6,13,13,100.00


**Interpretation:** The share of missing values is almost identical for the two main vendors (12.2% and 11.9%), so missing data is not a quirk of one provider. Vendor 6 has 13 trips and all of them lack passengers: they are all flex fare. Therefore `passenger_count IS NULL` effectively encodes the record type and duplicates `payment_type = 0` as a feature.

### Q4

**Business-question:** How much does a trip from an airport cost compared with other pickups?

In [142]:
query('''
SELECT
  z.zone AS pickup_zone,
  COUNT(*) AS trips,
  ROUND(AVG(t.trip_distance), 2) AS avg_distance,
  ROUND(AVG(t.total_amount), 2) AS avg_total,
  ROUND(AVG(COALESCE(t.airport_fee, 0)), 2) AS avg_airport_fee
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS z ON z.location_id = t.pu_location_id
WHERE z.zone ILIKE '%airport%'
GROUP BY z.zone
UNION ALL
SELECT
  'all other zones',
  COUNT(*),
  ROUND(AVG(t.trip_distance), 2),
  ROUND(AVG(t.total_amount), 2),
  ROUND(AVG(COALESCE(t.airport_fee, 0)), 2)
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS z ON z.location_id = t.pu_location_id
WHERE z.zone NOT ILIKE '%airport%' OR z.zone IS NULL
''')

,pickup_zone,trips,avg_distance,avg_total,avg_airport_fee
0,JFK Airport,4411,15.61,77.67,1.57
1,LaGuardia Airport,3011,9.50,65.88,1.70
2,Newark Airport,10,3.76,82.67,0.00
3,all other zones,92562,5.55,23.41,0.01


**Interpretation:** From JFK the sample has 4,411 trips with an average receipt of $77.67 and a distance of 15.6 miles, from LaGuardia 3,011 trips at $65.88. The other zones average $23.41. Airports account for 7.4% of trips but a large share of revenue, and `airport_fee` practically never occurs outside them, so the pickup zone is a strong feature for price prediction. Newark is almost absent (10 trips) because it is served mostly from New Jersey.

### Q5

**Business-question:** Which trips have a negative amount but are not marked as a dispute, no charge or voided?

In [144]:
query('''
SELECT
  trip_id,
  pickup_at,
  payment_type,
  trip_distance,
  fare_amount,
  total_amount
FROM hw3_taxi_trips
WHERE (fare_amount < 0 OR total_amount < 0)
  AND NOT (payment_type IN (3, 4, 6))
ORDER BY total_amount
LIMIT 10
''')

,trip_id,pickup_at,payment_type,trip_distance,fare_amount,total_amount
0,58552,2024-03-19 12:54:08+00:00,2,6.58,-200.00,-201.00
1,17550,2024-03-06 23:01:08+00:00,2,19.55,-99.30,-128.98
2,17441,2024-03-06 22:37:16+00:00,2,25.19,-97.50,-105.75
3,98475,2024-03-31 17:35:02+00:00,2,12.61,-85.00,-95.44
4,55459,2024-03-18 03:17:50+00:00,2,22.94,-86.30,-90.55
5,13899,2024-03-05 21:30:07+00:00,2,17.77,-70.00,-87.69
6,82524,2024-03-26 22:32:17+00:00,2,16.69,-70.00,-87.69
7,79279,2024-03-25 20:14:39+00:00,2,5.90,-70.00,-85.94
8,63094,2024-03-20 21:13:50+00:00,2,18.40,-77.90,-83.65
9,3796,2024-03-02 10:09:31+00:00,2,17.34,-77.90,-82.90


**Interpretation:** 665 rows have a negative amount although the payment is not marked as a dispute, no charge or voided; almost all of them are cash. The largest reversal is −$201 for 6.58 miles. So `payment_type` cannot be trusted as a marker of adjustments, and the sign of the amount (`is_negative_amount`) should be used for filtering.

### Q6

**Business-question:** How does the price per mile change for trips of 10 to 30 miles depending on the rate code?

In [146]:
query('''
SELECT
  COALESCE(rate_code_id::TEXT, 'unknown') AS rate_code,
  COUNT(*) AS trips,
  ROUND(AVG(fare_amount / NULLIF(trip_distance, 0)), 2) AS avg_fare_per_mile,
  ROUND(AVG(fare_amount), 2) AS avg_fare
FROM hw3_taxi_trips
WHERE trip_distance BETWEEN 10 AND 30
  AND fare_amount > 0
GROUP BY rate_code_id
ORDER BY trips DESC
''')

,rate_code,trips,avg_fare_per_mile,avg_fare
0,1,3269,4.27,56.00
1,2,2994,3.84,70.00
2,unknown,629,3.57,49.85
3,3,212,5.20,91.65
4,4,147,5.49,106.29
5,5,90,5.74,94.74


**Interpretation:** For trips of 10–30 miles the standard rate costs $4.27 per mile and the flat JFK rate $3.84, because the price there is the same regardless of distance (average fare exactly $70). The most expensive mile is in the negotiated rate ($5.74) and Nassau/Westchester ($5.49). 629 such trips have an unknown rate code, so `COALESCE` keeps them visible as a separate group instead of losing them.

### Q7

**Business-question:** What is the average tip percentage for card payments, and does it differ between short and long trips?

In [149]:
query('''
SELECT
  CASE WHEN trip_distance < 2 THEN 'under 2 miles' ELSE '2+ miles' END AS distance_group,
  COUNT(*) AS card_trips,
  ROUND(AVG(tip_amount * 100.0 / NULLIF(fare_amount, 0)), 2) AS avg_tip_pct,
  ROUND(COUNT(*) FILTER (WHERE tip_amount = 0) * 100.0 / COUNT(*), 2) AS pct_no_tip
FROM hw3_taxi_trips
WHERE payment_type = 1
  AND fare_amount > 0
GROUP BY distance_group
ORDER BY distance_group
''')

,distance_group,card_trips,avg_tip_pct,pct_no_tip
0,2+ miles,31491,22.07,6.71
1,under 2 miles,40612,28.19,4.34


**Interpretation:** For card payments, short trips (under 2 miles) get 28.2% tips on average, longer ones 22.1%. 4.3% and 6.7% of trips respectively get no tip. Apparently some passengers give a fixed amount, so as a percentage of the fare short trips look more generous. Cash tips are not recorded in the file, so tips can only be analyzed for `payment_type = 1`.

### Q8

**Business-question:** What share of trips ends in a different borough than where it started?

In [153]:
query('''
SELECT
  pu.borough AS pickup_borough,
  COUNT(*) AS trips,
  COUNT(*) FILTER (WHERE pu.borough IS DISTINCT FROM dz.borough) AS cross_borough_trips,
  ROUND(COUNT(*) FILTER (WHERE pu.borough IS DISTINCT FROM dz.borough) * 100.0 / COUNT(*), 2) AS pct_cross_borough
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pu ON pu.location_id = t.pu_location_id
JOIN hw3_taxi_zones AS dz ON dz.location_id = t.do_location_id
GROUP BY pu.borough
ORDER BY trips DESC
''')

,pickup_borough,trips,cross_borough_trips,pct_cross_borough
0,Manhattan,89191,5279,5.92
1,Queens,8845,6769,76.53
2,Brooklyn,1256,598,47.61
3,NaN,374,113,30.21
4,Bronx,314,187,59.55
5,EWR,10,3,30.00
6,Staten Island,4,3,75.00


**Interpretation:** 89,190 trips (89%) start in Manhattan, and only 5.9% of them leave its borders. From Queens, where the airports are, 76.6% of trips end in a different borough. `IS DISTINCT FROM` also counts trips into zones with an unknown borough (`NULL`), which a plain `<>` would skip: for the 374 trips from unknown zones it gives 30% "crossings".

### Q9

**Business-question:** How many distinct pickup zones and routes are actually present in the sample?

In [154]:
query('''
SELECT
  (SELECT COUNT(*) FROM hw3_taxi_zones) AS zones_in_lookup,
  (SELECT COUNT(DISTINCT pu_location_id) FROM hw3_taxi_trips) AS distinct_pickup_zones,
  (SELECT COUNT(DISTINCT do_location_id) FROM hw3_taxi_trips) AS distinct_dropoff_zones,
  (SELECT COUNT(*) FROM (SELECT DISTINCT pu_location_id, do_location_id FROM hw3_taxi_trips) AS pairs) AS distinct_routes
''')

,zones_in_lookup,distinct_pickup_zones,distinct_dropoff_zones,distinct_routes
0,265,230,246,7644


**Interpretation:** Of the 265 zones in the lookup, 230 occur as pickup points and 246 as drop-off points in the sample. There are 7,644 distinct routes for 99,994 trips, so the route distribution has a long tail. For a model, the zone pair is better encoded with target encoding or embeddings than with one-hot.

### Q10

**Business-question:** Which ten routes between zones are the most popular?

In [155]:
query('''
SELECT
  pu.zone AS pickup_zone,
  dz.zone AS dropoff_zone,
  COUNT(*) AS trips,
  ROUND(AVG(t.total_amount), 2) AS avg_total
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pu ON pu.location_id = t.pu_location_id
JOIN hw3_taxi_zones AS dz ON dz.location_id = t.do_location_id
WHERE t.pu_location_id IS DISTINCT FROM t.do_location_id
GROUP BY pu.zone, dz.zone
ORDER BY trips DESC
LIMIT 10
''')

,pickup_zone,dropoff_zone,trips,avg_total
0,Upper East Side South,Upper East Side North,649,14.67
1,Upper East Side North,Upper East Side South,522,15.75
2,Lincoln Square East,Upper West Side South,278,14.55
3,Midtown Center,Upper East Side South,270,16.79
4,Upper West Side South,Lincoln Square East,255,14.10
5,Upper East Side South,Midtown Center,254,16.42
6,Upper West Side South,Upper West Side North,248,13.31
7,Upper East Side North,Midtown Center,245,21.78
8,Upper East Side South,Midtown East,244,15.83
9,Midtown Center,Upper East Side North,222,20.30


**Interpretation:** All ten most popular routes (excluding trips within one zone) lie in Manhattan between the Upper East Side, Upper West Side, Lincoln Square and Midtown. Upper East Side South → North leads with 649 trips and an average receipt of $14.67. Short intra-city trips make up the bulk of demand, while airports bring the revenue per ride.

## 7. Reflection

**Quality problems.** The biggest one is a separate record type, flex fare (12%), in which five fields are empty at once. There is also the marker `RatecodeID = 99`, zero passengers, `Unknown`/`N/A` zones, reversals with negative amounts (even with `payment_type = 2`), six trips with drop-off before pickup, impossible distances up to 176 thousand miles, and meters left running for a whole day.

**Features.** Useful are pickup time (hour, weekday), pickup and drop-off zones, rate code, vendor and distance after removing outliers. I would not use `passenger_count`, because the driver types it and it is often 0 or `NULL`, `store_and_fwd` (a technical connectivity flag), as well as `trip_id` and `is_negative_amount`.

**Leakage.** If the model predicts price or tips at pickup time, `dropoff_at`, duration, the actual `trip_distance`, `tolls_amount`, `tip_amount`, `total_amount` and `payment_type` cannot be used. They become known only after the trip is finished. Only a route-based distance estimate may be an input.

**Pipeline.** I would ask the data engineer to pass an explicit trip id and a link from a reversal to its original receipt, write timestamps with a UTC offset, encode unknown values as `NULL` instead of `99` and `0`, and run the `dropoff >= pickup` validation and upper distance limits on the device itself.

**ML task.** Natural fits are regression (trip cost or duration), classification (whether the passenger leaves a tip) and demand forecasting by zone and hour. For the first model I would choose cost regression at pickup time, because it shows most clearly which fields are available before the trip and which are not.